# Turbofan Motorlarında Kalan Faydalı Ömür (RUL) Tahmini
## Bölüm 1 — Problem Tanımı, Veri Seti, EDA ve Veri Kalitesi (Aşama 1–4)

**Veri:** NASA C-MAPSS, FD001 alt kümesi · **Problem türü:** Regresyon (zaman serisi sensör verisi üzerinde) · **Ortam:** Google Colab

> ⚠️ **Sızıntı kuralı:** Bu notebook'ta yalnızca `train_FD001.txt` kullanılır. `test_FD001.txt` ve `RUL_FD001.txt` **Aşama 12'ye (final değerlendirme) kadar açılmayacaktır.** EDA, veri kalitesi kararları, ölçekleme ve tuning yalnızca eğitim verisine dayanır.

## Aşama 1 — Problem Tanımı ve Proje Planı

**Problem statement:** Uçak motorlarında plansız arıza hem güvenlik riski hem de yüksek maliyet demektir. Bakımı çok erken yapmak ise hâlâ kullanılabilir parçaların boşa harcanmasına yol açar. Bu projede, bir turbofan motorunun uçuş döngüleri boyunca kaydedilen 21 sensör ölçümü ve 3 çalışma ayarından yararlanarak motorun **arızaya kadar kaç uçuş döngüsü daha çalışabileceğini (Remaining Useful Life, RUL)** tahmin eden bir regresyon modeli geliştirilecektir. Sistem şu soruya cevap verir: *"Bu motorun şu ana kadarki sensör geçmişine bakıldığında, kaç döngü sonra bakıma alınmalı?"*

| Başlık | Tanım |
|---|---|
| **Problem türü** | Regresyon — çok değişkenli zaman serisi verisi üzerinde, motor (grup) bazlı |
| **Hedef değişken** | `RUL` = motorun son döngüsü − mevcut döngü (eğitim verisinden türetilir) |
| **Hedef kullanıcı** | Bakım planlama mühendisi / filo bakım yöneticisi (karar **destek** aracı) |
| **Kullanım senaryosu** | Mühendis bir motorun sensör kaydını yükler → model tahmini kalan ömrü ve risk seviyesini gösterir → kalan ömrü eşik altına düşen motor bakım planına alınır |
| **Öncelikli metrik** | **RMSE** (literatürle doğrudan karşılaştırılabilir) |
| **Yardımcı metrikler** | **NASA skor fonksiyonu** (ömrü *fazla* tahmin etmeyi daha ağır cezalandırır — güvenlik açısından tehlikeli hata budur), MAE, R² |
| **Başarı kriteri** | Resmi test setinde (100 motor) baseline Linear Regression'dan (~22 RMSE) anlamlı biçimde iyi olmak; hedef **RMSE ≈ 15–18** (klasik ML ile erken derin öğrenme çalışmalarına yakın seviye) |

**Proje planı:** Aşama 1–4 (bu notebook) → Aşama 5–6 preprocessing, feature engineering, GroupKFold deney tasarımı → Aşama 7–9 baseline ve model karşılaştırması → Aşama 10–11 tuning ve güçlü rakipler → Aşama 12 resmi test seti + literatür karşılaştırması → Aşama 13–14 Streamlit + deploy → Aşama 15 rapor ve sunum.

**Riskler:**
- **Veri sızıntısı:** Aynı motorun döngüleri hem eğitim hem doğrulama setine düşerse skorlar yapay olarak iyi çıkar → motor bazlı bölme (GroupKFold) zorunlu.
- **Ömrün başında bozulma görünmez:** İlk döngülerde sensörler neredeyse sabit; RUL'u bu bölgede tahmin etmek imkânsıza yakın → RUL üst sınırı (clipping) değerlendirilecek.
- **Veri simülasyondur:** Gerçek motor verisi değil; genelleme sınırlı.
- **Test seti küçük:** Resmi değerlendirme 100 motorun son döngüsü üzerinden yapılır → metriklerde varyans olabilir.

## Aşama 2 — Veri Setinin Bulunması ve Tanıtılması

- **Kaynak:** Saxena, A., Goebel, K., Simon, D., Eklund, N. (2008). *Damage Propagation Modeling for Aircraft Engine Run-to-Failure Simulation.* PHM 2008. — NASA Ames Prognostics Center of Excellence.
- **Erişim:** Kaggle — *NASA Turbofan Jet Engine Data Set* (C-MAPSS) · NASA Open Data Portal: https://data.nasa.gov/dataset/cmapss-jet-engine-simulated-data
- **Lisans / kullanım koşulları:** NASA açık veri portalında kamuya açık olarak yayımlanmıştır; kaynak gösterilerek kullanılmalıdır. *(Rapora eklemeden önce Kaggle ve NASA sayfasındaki lisans ifadesini kontrol edip buraya yazın.)*
- **Gizlilik:** Simülasyon verisidir; kişisel veri içermez.

**Dosya yapısı (FD001: tek çalışma koşulu — deniz seviyesi, tek arıza tipi — HPC bozulması):**

| Dosya | İçerik | Bu projede kullanımı |
|---|---|---|
| `train_FD001.txt` | 100 motorun arızaya kadar **tam** kayıtları | Eğitim + GroupKFold validation |
| `test_FD001.txt` | 100 farklı motorun arızadan önce **kesilmiş** kayıtları | Yalnızca Aşama 12 |
| `RUL_FD001.txt` | Her test motorunun kesildiği andaki gerçek RUL değeri (100 sayı) | Yalnızca Aşama 12 |

Dosyalar boşlukla ayrılmış, başlık satırı olmayan metin dosyalarıdır; sütun adlarını biz veriyoruz.

## 0. Kurulum ve veri yükleme

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
np.random.seed(SEED)
sns.set_theme(style="whitegrid", palette="tab10")
pd.set_option("display.max_columns", None)

# Colab: train_FD001.txt dosyasını sol paneldeki 'Files' bölümüne yükleyin
# veya: from google.colab import files; files.upload()
TRAIN_PATH = "train_FD001.txt"

INDEX_COLS = ["unit", "cycle"]
SETTING_COLS = ["op1", "op2", "op3"]
SENSOR_COLS = [f"s{i}" for i in range(1, 22)]
COLS = INDEX_COLS + SETTING_COLS + SENSOR_COLS

train = pd.read_csv(TRAIN_PATH, sep=r"\s+", header=None)
print("Ham sütun sayısı:", train.shape[1])  # 26 bekleniyor (bazı sürümlerde sondaki boşluklar 2 boş sütun üretir)
train = train.iloc[:, :26]
train.columns = COLS
print("Boyut:", train.shape)

### Veri sözlüğü

In [ ]:
sensor_info = pd.DataFrame([
    ("s1",  "T2",        "Fan girişi toplam sıcaklık",            "°R"),
    ("s2",  "T24",       "LPC çıkışı toplam sıcaklık",            "°R"),
    ("s3",  "T30",       "HPC çıkışı toplam sıcaklık",            "°R"),
    ("s4",  "T50",       "LPT çıkışı toplam sıcaklık",            "°R"),
    ("s5",  "P2",        "Fan girişi basınç",                     "psia"),
    ("s6",  "P15",       "Bypass kanalı toplam basınç",           "psia"),
    ("s7",  "P30",       "HPC çıkışı toplam basınç",              "psia"),
    ("s8",  "Nf",        "Fiziksel fan hızı",                     "rpm"),
    ("s9",  "Nc",        "Fiziksel çekirdek hızı",                "rpm"),
    ("s10", "epr",       "Motor basınç oranı (P50/P2)",           "–"),
    ("s11", "Ps30",      "HPC çıkışı statik basınç",              "psia"),
    ("s12", "phi",       "Yakıt akışı / Ps30 oranı",              "pps/psi"),
    ("s13", "NRf",       "Düzeltilmiş fan hızı",                  "rpm"),
    ("s14", "NRc",       "Düzeltilmiş çekirdek hızı",             "rpm"),
    ("s15", "BPR",       "Bypass oranı",                          "–"),
    ("s16", "farB",      "Yanma odası yakıt-hava oranı",          "–"),
    ("s17", "htBleed",   "Tahliye (bleed) entalpisi",             "–"),
    ("s18", "Nf_dmd",    "İstenen fan hızı",                      "rpm"),
    ("s19", "PCNfR_dmd", "İstenen düzeltilmiş fan hızı",          "rpm"),
    ("s20", "W31",       "HPT soğutma havası tahliyesi",          "lbm/s"),
    ("s21", "W32",       "LPT soğutma havası tahliyesi",          "lbm/s"),
], columns=["sütun", "sembol", "açıklama", "birim"])

other_info = pd.DataFrame([
    ("unit", "–", "Motor numarası (grup kimliği — özellik olarak KULLANILMAYACAK)", "–"),
    ("cycle", "–", "Uçuş döngüsü sırası", "döngü"),
    ("op1–op3", "–", "Çalışma ayarları (irtifa, Mach, gaz kolu açısı ile ilişkili)", "–"),
    ("RUL", "–", "HEDEF: kalan faydalı ömür (türetilecek)", "döngü"),
], columns=["sütun", "sembol", "açıklama", "birim"])
display(pd.concat([other_info, sensor_info], ignore_index=True))

*Kısaltmalar: LPC/HPC = alçak/yüksek basınç kompresörü, LPT/HPT = alçak/yüksek basınç türbini. Kaynak: Saxena vd. (2008), Tablo 2.*

### Gözlem/değişken sayısı, veri tipleri ve hedef değişkenin oluşturulması

In [ ]:
print(f"Satır (döngü) sayısı : {len(train)}")
print(f"Motor sayısı         : {train['unit'].nunique()}")
print(f"Özellik sayısı       : {len(SETTING_COLS) + len(SENSOR_COLS)} (3 ayar + 21 sensör)")
train.info()

# Hedef: her motorun son döngüsünde RUL = 0
max_cycle = train.groupby("unit")["cycle"].transform("max")
train["RUL"] = max_cycle - train["cycle"]
display(train.head())

**Not:** `RUL` hedefi eğitim verisinde motorun arıza anına kadar kayıtlı olmasından türetilir. Her motor kendi içinde hesaplandığı için bu adım sızıntı oluşturmaz.

**Veri setinin sınırlılıkları:** (1) Simülasyon verisi; gerçek motorlardaki bakım müdahaleleri, sensör arızaları ve çevresel çeşitlilik yok. (2) FD001 tek çalışma koşulu ve tek arıza tipi içerir; sonuçlar FD002–FD004'e doğrudan genellenemez. (3) 100 motor gibi küçük bir filo; motorlar arası çeşitlilik sınırlı. (4) Sensörlere gürültü eklenmiştir.

## Aşama 3 — Keşifsel Veri Analizi (EDA)
### 3.1 Head / Tail ve genel özet

In [ ]:
display(train.head()); display(train.tail())

In [ ]:
display(train.describe().T.round(4))

**İlk gözlem:** Bazı sütunların (`op3`, `s1`, `s5`, `s10`, `s16`, `s18`, `s19`) standart sapması 0 — tüm motorlarda tek değer alıyorlar. Bunlar Aşama 4'te sistematik olarak incelenecek.

### 3.2 Motor ömürlerinin dağılımı

In [ ]:
life = train.groupby("unit")["cycle"].max()
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(life, bins=20, kde=True, ax=ax[0]); ax[0].set_title("Motor ömrü (toplam döngü) dağılımı"); ax[0].set_xlabel("döngü")
sns.boxplot(x=life, ax=ax[1]); ax[1].set_title("Motor ömrü — boxplot"); ax[1].set_xlabel("döngü")
plt.tight_layout(); plt.show()
print(life.describe().round(1))

**Yorum:** Motorlar 128 ile 362 döngü arasında çalışıp arızalanmış; ortalama ~206 döngü. Ömür dağılımı sağa çarpık — birkaç motor diğerlerinden çok daha uzun yaşıyor. Bu, her motorun farklı başlangıç aşınması ve üretim farkıyla başladığını gösterir.

### 3.3 Hedef değişken (RUL) dağılımı

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(train["RUL"], bins=40, ax=ax); ax.set_title("RUL dağılımı (tüm döngüler)"); ax.set_xlabel("RUL (döngü)")
plt.show()
print("Çarpıklık:", round(train["RUL"].skew(), 2))

**Yorum:** Düşük RUL değerleri tüm motorlarda bulunduğu için yoğun; yüksek RUL değerleri yalnızca uzun ömürlü motorlardan geliyor. Hedef sağa çarpık.

### 3.4 Sensörlerin zaman içindeki seyri (örnek motorlar)

In [ ]:
sample_units = [1, 25, 50, 75, 100]
fig, axes = plt.subplots(7, 3, figsize=(18, 24), sharex=True)
for ax, s in zip(axes.flat, SENSOR_COLS):
    for u in sample_units:
        d = train[train["unit"] == u]
        ax.plot(d["RUL"], d[s], lw=0.8, label=f"motor {u}")
    ax.set_title(f"{s} — {sensor_info.set_index('sütun').loc[s, 'sembol']}")
    ax.invert_xaxis()
axes.flat[0].legend(fontsize=8)
for ax in axes[-1]: ax.set_xlabel("RUL (arızaya kalan döngü) →  0")
plt.tight_layout(); plt.show()

**Yorum:** X ekseni ters çevrildi; sağa doğru arıza anına yaklaşılıyor.
- `s2, s3, s4, s8, s11, s13, s15, s17` arızaya yaklaştıkça **artıyor**; `s7, s12, s20, s21` **azalıyor** → bozulmayı taşıyan sensörler.
- `s9` ve `s14` motorlar arasında farklı yönlerde seyrediyor → tek başına daha zayıf gösterge.
- `s1, s5, s10, s16, s18, s19` düz çizgi → bilgi taşımıyor.
- Tüm sensörlerde belirgin **gürültü** var; trend ancak yumuşatılınca netleşiyor (Aşama 4.6).

### 3.5 Bozulmanın ömür boyunca seyri — normalize edilmiş zaman

In [ ]:
tmp = train.copy()
tmp["life_frac"] = tmp["cycle"] / tmp.groupby("unit")["cycle"].transform("max")  # 0 = başlangıç, 1 = arıza
tmp["bin"] = (tmp["life_frac"] * 20).clip(upper=19.999).astype(int) / 20
key = ["s2", "s3", "s4", "s7", "s11", "s12", "s15", "s21"]
z = tmp.groupby("bin")[key].mean()
z = (z - z.iloc[0]) / train[key].std()  # başlangıca göre standart sapma cinsinden değişim
z.plot(figsize=(11, 5), marker="o", ms=3)
plt.axhline(0, c="k", lw=0.6); plt.xlabel("Ömür oranı (0 = başlangıç, 1 = arıza)"); plt.ylabel("Başlangıca göre değişim (std)")
plt.title("Sensörlerde ortalama bozulma eğrisi"); plt.show()

**Kritik bulgu:** Ömrün ilk ~%40'ında sensörlerdeki ortalama değişim başlangıca göre 0.3 standart sapmanın altında kalıyor; son %10'da ise 2–2.6 standart sapmaya çıkıyor. Bozulma ömrün ikinci yarısında hızlanarak (üstel benzeri) ortaya çıkıyor. Bu şu anlama gelir: ömrün başındaki bir motorun RUL'unun 300 mü 150 mi olduğunu sensörlerden ayırt etmek **mümkün değil**. Bu nedenle literatürde yaygın olan **parçalı doğrusal RUL etiketlemesi** (RUL'u bir üst değerde, ör. 125'te sabitlemek) Aşama 5'te değerlendirilecek.

### 3.6 Sensör–RUL ilişkisi (korelasyon)

In [ ]:
corr_rul = train[SETTING_COLS + SENSOR_COLS + ["RUL"]].corr()["RUL"].drop("RUL").dropna().sort_values()
fig, ax = plt.subplots(figsize=(9, 5))
corr_rul.plot(kind="barh", ax=ax, color=["tab:red" if v < 0 else "tab:blue" for v in corr_rul])
ax.set_title("Özelliklerin RUL ile Pearson korelasyonu"); ax.axvline(0, c="k", lw=0.6)
plt.show()

# Spearman: monoton ama doğrusal olmayan ilişkileri de yakalar
spear = train[SENSOR_COLS + ["RUL"]].corr(method="spearman")["RUL"].drop("RUL").dropna()
display(pd.DataFrame({"pearson": corr_rul, "spearman": spear}).dropna().sort_values("pearson").round(3))

**Yorum:** En güçlü ilişkiler `s11` (Ps30), `s4` (T50), `s12` (phi), `s7` (P30), `s15` (BPR) ve `s21` (W32) sensörlerinde (|r| ≈ 0.63–0.70). Çalışma ayarlarının (`op1`, `op2`) RUL ile ilişkisi ~0 — FD001'de tek çalışma koşulu olduğu için beklenen bir sonuç. Korelasyon değerleri, ilişkinin doğrusal olmadığını da düşündürüyor (bozulma ömür sonunda hızlanıyor).

### 3.7 Sensörler arası korelasyon (çoklu doğrusallık)

In [ ]:
informative = [s for s in SENSOR_COLS if train[s].std() > 1e-3]
plt.figure(figsize=(11, 9))
sns.heatmap(train[informative].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, annot_kws={"size": 7})
plt.title("Bilgi taşıyan sensörler arası korelasyon"); plt.show()

c = train[informative].corr().abs()
pairs = (c.where(np.triu(np.ones(c.shape), k=1).astype(bool)).stack().sort_values(ascending=False))
print("En yüksek korelasyonlu sensör çiftleri:"); print(pairs.head(6).round(3))

**Yorum:** `s9` (Nc) ve `s14` (NRc) neredeyse aynı bilgiyi taşıyor (fiziksel ve düzeltilmiş çekirdek hızı). Bozulma sensörlerinin çoğu kendi aralarında orta-yüksek korelasyonlu — hepsi aynı bozulma sürecini farklı açılardan ölçüyor. Doğrusal modellerde çoklu doğrusallık katsayı yorumunu zorlaştırabilir; Ridge (L2) gibi düzenlileştirme bu nedenle değerlendirilecek.

### 3.8 Çok değişkenli görünüm

In [ ]:
pp = train[train["unit"].isin(range(1, 101, 4))][["s4", "s11", "s12", "s21", "RUL"]]
sns.pairplot(pp, hue="RUL", palette="viridis", corner=True, plot_kws={"s": 6, "alpha": 0.5, "linewidth": 0})
plt.show()

**Yorum:** Düşük RUL (koyu renk) noktaları, sensör uzayında belirgin bir köşeye kayıyor; sensörler birlikte kullanıldığında bozulma durumu tek bir sensöre göre çok daha net ayrışıyor → çok değişkenli modellerin avantajı.

## Aşama 4 — Veri Kalitesi Değerlendirmesi
### 4.1 Eksik değerler

In [ ]:
print("Toplam eksik değer:", train.isna().sum().sum())
print("Sonsuz değer:", np.isinf(train.select_dtypes("number")).sum().sum())

### 4.2 Tekrarlanan kayıtlar ve döngü sürekliliği

In [ ]:
print("Tam tekrar eden satır:", train.duplicated().sum())
print("(unit, cycle) çifti tekrarı:", train.duplicated(subset=["unit", "cycle"]).sum())

gaps = train.groupby("unit")["cycle"].apply(lambda s: int((s.diff().dropna() != 1).sum()))
starts = train.groupby("unit")["cycle"].min()
print("Döngü atlaması/bozuk sıra olan motor sayısı:", (gaps > 0).sum())
print("1'den başlamayan motor sayısı:", (starts != 1).sum())

**Bulgu:** Eksik değer, tekrar eden kayıt, atlanmış döngü yok; her motor 1. döngüden başlıyor. Zaman serileri bütünlüklü.

### 4.3 Sabit ve neredeyse sabit sütunlar

In [ ]:
q = pd.DataFrame({
    "benzersiz_değer": train[SETTING_COLS + SENSOR_COLS].nunique(),
    "std": train[SETTING_COLS + SENSOR_COLS].std(),
    "RUL_korelasyon": train[SETTING_COLS + SENSOR_COLS].corrwith(train["RUL"]),
}).sort_values("std")
display(q.round(5))

CONSTANT_COLS = q.index[q["benzersiz_değer"] <= 1].tolist()
NEAR_CONSTANT = q.index[(q["benzersiz_değer"] > 1) & (q["benzersiz_değer"] <= 3)].tolist()
print("Tam sabit sütunlar        :", CONSTANT_COLS)
print("Neredeyse sabit (≤3 değer):", NEAR_CONSTANT)

**Bulgu:**
- `op3, s1, s5, s10, s16, s18, s19` → **tam sabit**; hiçbir bilgi taşımıyorlar → **çıkarılacak**.
- `s6` → yalnızca 2 farklı değer alıyor (21.60 / 21.61) ve RUL ile korelasyonu zayıf; sensör çözünürlüğünden kaynaklanan bir kuantalama → **çıkarılacak** (literatürle de uyumlu).
- `op1`, `op2` → değişkenlik çok küçük ve RUL ile ilişkisiz (tek çalışma koşulu) → **çıkarılacak**.
- Sonuç: **14 sensör** (`s2, s3, s4, s7, s8, s9, s11, s12, s13, s14, s15, s17, s20, s21`) modellemeye taşınacak.

### 4.4 Veri tipleri ve ayrık sensörler

In [ ]:
for s in ["s17", "s8", "s13"]:
    print(f"{s}: {train[s].nunique()} benzersiz değer, örnek: {sorted(train[s].unique())[:8]}")

**Bulgu:** `s17` (htBleed) tamsayı değerler alıyor (388–400); sensör çözünürlüğü düşük ama trend taşıyor, sayısal olarak tutulacak. `s8` ve `s13` ise 0.01 hassasiyetle kaydedilmiş; bu da küçük değişimlerin basamaklı görünmesine yol açıyor. Tip dönüşümü gerektiren sütun yok.

### 4.5 Aykırı değer (outlier) incelemesi

In [ ]:
FEATURES_14 = ["s2", "s3", "s4", "s7", "s8", "s9", "s11", "s12", "s13", "s14", "s15", "s17", "s20", "s21"]
rows = []
for s in FEATURES_14:
    q1, q3 = train[s].quantile([.25, .75]); iqr = q3 - q1
    out = (train[s] < q1 - 1.5 * iqr) | (train[s] > q3 + 1.5 * iqr)
    rows.append([s, int(out.sum()), round(out.mean() * 100, 2),
                 round(train.loc[out, "RUL"].median(), 1) if out.any() else np.nan])
out_df = pd.DataFrame(rows, columns=["sensör", "IQR_outlier", "oran_%", "outlier_RUL_medyanı"])
display(out_df)
print("Tüm verinin RUL medyanı:", train["RUL"].median())

**Yorum:** IQR dışında kalan değerlerin RUL medyanı genel medyandan çok düşük → bu "aykırı" değerler büyük ölçüde **arızaya yakın döngülerden** geliyor. Yani ölçüm hatası değil, tam olarak yakalamak istediğimiz **bozulma sinyali**. Silinmeleri modelin en kritik bilgiyi kaybetmesine neden olur → **outlier'lar silinmeyecek.**

### 4.6 Sensör gürültüsü

In [ ]:
u = train[train["unit"] == 1]
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, s in zip(axes, ["s4", "s11", "s21"]):
    ax.plot(u["cycle"], u[s], alpha=0.4, label="ham")
    ax.plot(u["cycle"], u[s].rolling(10, min_periods=1).mean(), lw=2, label="10 döngü kayan ortalama")
    ax.set_title(f"Motor 1 — {s}"); ax.set_xlabel("döngü"); ax.legend()
plt.tight_layout(); plt.show()

# Gürültü oranı: döngüden döngüye fark (yüksek frekans) / toplam değişim aralığı
noise = {s: train.groupby("unit")[s].diff().std() / (train[s].max() - train[s].min()) for s in FEATURES_14}
display(pd.Series(noise, name="gürültü_oranı").sort_values(ascending=False).round(3).to_frame())

**Bulgu:** Ham sinyallerde döngüden döngüye belirgin rastgele dalgalanma var; trend ancak kayan ortalama ile netleşiyor. Bu durum Aşama 5'te **motor bazında kayan pencere özellikleri** (ortalama, standart sapma, eğim) üretmek için güçlü bir gerekçe. Pencere hesapları **her motor içinde** yapılmalı; aksi halde bir motorun verisi bir sonrakine sızar.

### 4.7 Sınıf dengesizliği yerine: hedef dağılımının dengesizliği

In [ ]:
for cap in [None, 150, 125, 100]:
    y = train["RUL"] if cap is None else train["RUL"].clip(upper=cap)
    share = (y == cap).mean() * 100 if cap else 0
    print(f"Üst sınır={str(cap):>4} | ortalama={y.mean():6.1f} | std={y.std():5.1f} | çarpıklık={y.skew():5.2f} | sabitlenen satır=%{share:4.1f}")

**Yorum:** Regresyon olduğu için sınıf dengesizliği yok; ancak hedef sağa çarpık ve yüksek RUL değerleri yalnızca birkaç uzun ömürlü motordan geliyor. RUL'u 125'te sabitlemek satırların ~%39'unu etkiliyor ve standart sapmayı 69'dan 42'ye düşürüyor. Hangi üst sınırın kullanılacağı Aşama 5–6'da **yalnızca eğitim verisi üzerinde CV ile** seçilecek (literatürde 125 yaygın).

### 4.8 Grup yapısı — veri sızıntısı riski

In [ ]:
print("Motor başına döngü sayısı (min / medyan / max):", life.min(), life.median(), life.max())
print("s11 sensörünün motor içi ortalama otokorelasyonu (lag-1):",
      round(train.groupby("unit")["s11"].apply(lambda s: s.autocorr(1)).mean(), 3))

**Bulgu:** Her motor ortalama ~206 ardışık ve birbirine çok benzeyen satırdan oluşuyor. Satırlar rastgele bölünürse, modelin doğrulama setinde gördüğü döngülerin hemen öncesi ve sonrası eğitim setinde yer alır → skorlar yapay olarak iyi çıkar. **Karar:** Tüm validation işlemleri `GroupKFold(groups=unit)` ile motor bazında yapılacak; `unit` ve `cycle` sütunları **özellik olarak kullanılmayacak** (cycle, test setinde kesilme noktası rastgele olduğu için yanıltıcıdır).

### 4.9 Veri kalitesi özeti ve modellemeye etkisi

| Kontrol | Bulgu | Karar (Aşama 5'te uygulanacak) |
|---|---|---|
| Eksik / sonsuz değer | Yok | — |
| Tekrar eden satır, döngü atlaması | Yok; tüm motorlar 1. döngüden başlıyor | — |
| Tam sabit sütunlar | `op3, s1, s5, s10, s16, s18, s19` | Çıkar |
| Neredeyse sabit / ilişkisiz | `s6` (2 değer), `op1, op2` (tek çalışma koşulu) | Çıkar |
| Çoklu doğrusallık | `s9–s14` çok yüksek korelasyonlu | Ridge ile kontrol; ağaç modellerinde sorun değil |
| Outlier | Arızaya yakın döngülerden gelen gerçek bozulma sinyali | **Silinmeyecek** |
| Gürültü | Tüm sensörlerde belirgin | Motor bazlı kayan pencere özellikleri |
| Hedef dağılımı | Sağa çarpık; ömrün başında bozulma görünmüyor | Parçalı doğrusal RUL (üst sınır CV ile seçilecek) |
| Grup yapısı | Motor başına ~206 ardışık, ilişkili satır | `GroupKFold`; `unit` ve `cycle` özellik değil |
| Ayrık sensörler | `s17` tamsayı | Sayısal olarak tut |

**Sonraki adım (Bölüm 2):** 14 sensörle preprocessing ve feature engineering pipeline'ı, RUL üst sınırının seçimi, GroupKFold deney tasarımı ve baseline modeller.

### Ek: Sonraki notebook için seçilen özellikleri kaydet

In [ ]:
import json
DROP_COLS = ["op1", "op2", "op3", "s1", "s5", "s6", "s10", "s16", "s18", "s19"]
config = {"features": FEATURES_14, "drop": DROP_COLS, "seed": SEED}
with open("project_config.json", "w") as f:
    json.dump(config, f, indent=2)
print(config)